# LeetCode #133 — Clone Graph: sao chép sâu đồ thị với hash map "cũ → mới"

**Mức độ:** Medium · **Chủ đề:** Hash Table, Depth-First Search, Breadth-First Search, Graph

Skeleton trên LeetCode:

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val = 0, neighbors = None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []
"""
class Solution:
    def cloneGraph(self, node: Optional['Node']) -> Optional['Node']:
```

Mục tiêu của notebook:

1. Hiểu **sao chép sâu (deep copy)** đồ thị: mọi nút mới, cạnh giống hệt, **không** trỏ về nút cũ.
2. Hash map `old → new` vừa là "đã thăm" vừa là bảng tra để nối cạnh — chìa khóa xử lý **chu trình**.
3. Cài đặt bằng DFS (đệ quy) và BFS (hàng đợi).
4. Chuyển đổi giữa danh sách kề (adjacency list) và đồ thị nút để kiểm tra.

# 1. Bài toán là gì?

Cho tham chiếu tới **một nút** của một đồ thị **vô hướng, liên thông**. Trả về **bản sao sâu** của đồ thị. Mỗi nút có `val` (int) và `neighbors` (list các nút kề).

LeetCode biểu diễn đồ thị bằng **danh sách kề** đánh số từ 1: `adjList[i]` là danh sách láng giềng của nút có `val = i + 1`. Nút được cho luôn là nút `val = 1`.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 100]`.
- `1 <= Node.val <= 100`, mọi `val` **phân biệt**.
- Không có cạnh lặp, không có khuyên (self-loop).
- Đồ thị liên thông.

---

## Ví dụ

```text
adjList = [[2, 4], [1, 3], [2, 4], [1, 3]]

    1 --- 2
    |     |           ->  bản sao có cùng danh sách kề
    4 --- 3

adjList = [[]]   ->  [[]]      (một nút, không cạnh)
adjList = []     ->  []        (đồ thị rỗng, node = None)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Đồ thị (graph) và cách biểu diễn
2. Bảng băm (hash table) — `dict` và `set`
3. Duyệt theo chiều rộng (BFS — Breadth-First Search)
4. Duyệt theo chiều sâu (DFS — Depth-First Search)

Cuối mục có phần **2.5 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Đồ thị (graph) và cách biểu diễn

### Đồ thị là gì?

Đồ thị gồm các **đỉnh** (vertex/node) và các **cạnh** (edge) nối cặp đỉnh. Cạnh có thể **vô hướng** (đi được hai chiều, như đường hai chiều) hoặc **có hướng**. Cây là trường hợp riêng: đồ thị liên thông, không chu trình.

```text
1 ── 2
│    │       đỉnh: 1, 2, 3, 4
4 ── 3       cạnh: (1,2) (2,3) (3,4) (4,1)  → có chu trình 1-2-3-4-1
```

### Ba cách biểu diễn

| Cách | Mô tả | Hợp khi |
|---|---|---|
| Danh sách cạnh | `[(1,2), (2,3), …]` | đầu vào thô |
| **Danh sách kề** | `{1: [2, 4], 2: [1, 3], …}` | hầu hết bài duyệt (BFS/DFS) |
| Ma trận kề | `M[u][v] = 1` nếu có cạnh | đồ thị dày, cần hỏi "có cạnh u-v?" nhanh |
| Đối tượng nút | `node.neighbors = [...]` | LeetCode kiểu "Clone Graph" |

In [1]:
from collections import defaultdict, deque

canh = [(1, 2), (2, 3), (3, 4), (4, 1)]
ke = defaultdict(list)
for u, v in canh:                # vô hướng: thêm cả hai chiều
    ke[u].append(v)
    ke[v].append(u)
print("danh sách kề:", dict(ke))

n = 4
M = [[0] * (n + 1) for _ in range(n + 1)]
for u, v in canh:
    M[u][v] = M[v][u] = 1
print("ma trận kề (bỏ hàng/cột 0):")
for row in M[1:]:
    print("  ", row[1:])

danh sách kề: {1: [2, 4], 2: [1, 3], 3: [2, 4], 4: [3, 1]}
ma trận kề (bỏ hàng/cột 0):
   [0, 1, 0, 1]
   [1, 0, 1, 0]
   [0, 1, 0, 1]
   [1, 0, 1, 0]


### Đồ thị dạng đối tượng nút

Mỗi nút giữ danh sách **tham chiếu** tới các nút hàng xóm. Vì có chu trình, khi duyệt **bắt buộc** phải nhớ nút đã thăm — nếu không sẽ đi vòng mãi.

In [2]:
class GNode:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []

nodes = {i: GNode(i) for i in range(1, 5)}
for u, v in canh:
    nodes[u].neighbors.append(nodes[v])
    nodes[v].neighbors.append(nodes[u])

def duyet_nut(start):
    seen, q, out = {start}, deque([start]), []
    while q:
        x = q.popleft()
        out.append(x.val)
        for y in x.neighbors:
            if y not in seen:
                seen.add(y); q.append(y)
    return out

print("BFS từ nút 1:", duyet_nut(nodes[1]))
print("hàng xóm của 3:", [y.val for y in nodes[3].neighbors])

BFS từ nút 1: [1, 2, 4, 3]
hàng xóm của 3: [2, 4]


### Sao chép **nông** và sao chép **sâu**

- Sao chép nông: tạo đối tượng mới nhưng vẫn **trỏ tới** các nút cũ.
- Sao chép sâu: tạo **mọi** nút mới, và cạnh mới nối các nút **mới** với nhau; không chia sẻ gì với bản gốc.

Với đồ thị có chu trình, sao chép sâu cần một **bảng `cũ → mới`** để mỗi nút cũ chỉ được tạo bản sao **một lần** và các cạnh trỏ đúng bản sao.

In [3]:
import copy
a_list = [[1, 2], [3]]
nong = list(a_list)
sau = copy.deepcopy(a_list)
a_list[0].append(99)
print("gốc:", a_list, "| nông (bị ảnh hưởng):", nong, "| sâu:", sau)

gốc: [[1, 2, 99], [3]] | nông (bị ảnh hưởng): [[1, 2, 99], [3]] | sâu: [[1, 2], [3]]


## 2.2 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [4]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [5]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 242.50 ms | trên set: 0.110 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [6]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [7]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.3 Duyệt theo chiều rộng (BFS — Breadth-First Search)

### BFS là gì?

BFS duyệt đồ thị/cây **theo từng lớp**: thăm điểm xuất phát, rồi **mọi** điểm cách 1 bước, rồi mọi điểm cách 2 bước, … giống vết sóng lan ra khi ném hòn đá xuống nước.

Công cụ: **hàng đợi** (FIFO). Điểm phát hiện trước được xử lý trước → các điểm gần luôn được xử lý trước các điểm xa.

```text
        A            lớp 0: A
       / \           lớp 1: B, C
      B   C          lớp 2: D, E, F
     / \   \
    D   E   F        thứ tự BFS: A B C D E F
```

### Khung BFS trên đồ thị

```text
q = deque([start]); visited = {start}
while q:
    u = q.popleft()
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            visited.add(v)        # đánh dấu NGAY khi đưa vào hàng đợi
            q.append(v)
```

In [8]:
from collections import deque

do_thi = {                       # danh sách kề
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def bfs(graph, start):
    q, visited, thu_tu = deque([start]), {start}, []
    while q:
        u = q.popleft()
        thu_tu.append(u)
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
        print(f"lấy {u} | hàng đợi: {list(q)}")
    return thu_tu

print("Thứ tự BFS:", bfs(do_thi, "A"))

lấy A | hàng đợi: ['B', 'C']
lấy B | hàng đợi: ['C', 'D', 'E']
lấy C | hàng đợi: ['D', 'E', 'F']
lấy D | hàng đợi: ['E', 'F']
lấy E | hàng đợi: ['F']
lấy F | hàng đợi: []
Thứ tự BFS: ['A', 'B', 'C', 'D', 'E', 'F']


### BFS theo tầng (level-order) trên cây nhị phân

Muốn biết nút nào thuộc tầng nào: ở đầu mỗi vòng, hàng đợi chứa **đúng** các nút của một tầng. Ghi lại `len(q)`, lấy ra đúng chừng ấy nút (và đẩy con của chúng vào) — đó là một tầng.

In [9]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_bfs = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))

def theo_tang(root):
    if not root:
        return []
    q, tang = deque([root]), []
    while q:
        cur = []
        for _ in range(len(q)):          # đúng số nút của tầng hiện tại
            node = q.popleft()
            cur.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        tang.append(cur)
    return tang

print("các tầng:", theo_tang(cay_bfs))

các tầng: [[3], [9, 20], [15, 7]]


### BFS tìm đường ngắn nhất trên lưới

Trên đồ thị **không trọng số** (mỗi bước dài như nhau), lần đầu BFS chạm tới một ô chính là đi bằng **số bước ít nhất**. Lưu `dist` khi đưa ô vào hàng đợi.

In [10]:
luoi = ["S.#.",
        "..#.",
        "...E"]

def ngan_nhat(grid):
    R, C = len(grid), len(grid[0])
    start = next((r, c) for r in range(R) for c in range(C) if grid[r][c] == "S")
    dist = {start: 0}
    q = deque([start])
    while q:
        r, c = q.popleft()
        if grid[r][c] == "E":
            return dist[(r, c)]
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return -1

print("số bước ít nhất từ S tới E:", ngan_nhat(luoi))

số bước ít nhất từ S tới E: 5


### Độ phức tạp và lỗi hay gặp

`O(V + E)` thời gian (mỗi đỉnh vào hàng đợi một lần, mỗi cạnh xét một/hai lần), `O(V)` bộ nhớ. Trên cây: `O(n)`; trên lưới `R×C`: `O(R·C)`.

- Đánh dấu `visited` khi **lấy ra** thay vì khi **đưa vào** → một ô có thể vào hàng đợi nhiều lần (chậm, đếm sai).
- Dùng `list.pop(0)` → mỗi lần `O(n)`.
- Dùng `pop()` (cuối) → thành DFS, mất tính "gần trước".
- Quên kiểm tra biên lưới.

## 2.4 Duyệt theo chiều sâu (DFS — Depth-First Search)

### DFS là gì?

DFS đi **sâu nhất có thể** theo một nhánh, tới khi hết đường thì **quay lui** về ngã rẽ gần nhất và thử nhánh khác — giống đi mê cung bằng cách luôn bám một bức tường.

```text
        A
       / \          DFS từ A (thử hàng xóm theo thứ tự):
      B   C         A → B → D (cụt, quay lui) → E → F → C
     / \   \
    D   E - F
```

Công cụ: **đệ quy** (dùng call stack) hoặc **ngăn xếp** tự quản lý.

### Khung DFS

```text
def dfs(u):
    visited.add(u)
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            dfs(v)
```

In [11]:
do_thi_dfs = {
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def dfs_de_quy(graph, u, visited, thu_tu, sau=0):
    visited.add(u)
    thu_tu.append(u)
    print("    " * sau + f"vào {u}")
    for v in graph[u]:
        if v not in visited:
            dfs_de_quy(graph, v, visited, thu_tu, sau + 1)
    print("    " * sau + f"rời {u}")

tt = []
dfs_de_quy(do_thi_dfs, "A", set(), tt)
print("Thứ tự DFS:", tt)

vào A
    vào B
        vào D
        rời D
        vào E
            vào F
                vào C
                rời C
            rời F
        rời E
    rời B
rời A
Thứ tự DFS: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS bằng ngăn xếp (không đệ quy)

Thay hàng đợi của BFS bằng **ngăn xếp**. Tránh được giới hạn độ sâu đệ quy khi dữ liệu lớn.

In [12]:
def dfs_ngan_xep(graph, start):
    st, visited, thu_tu = [start], set(), []
    while st:
        u = st.pop()
        if u in visited:
            continue
        visited.add(u)
        thu_tu.append(u)
        for v in reversed(graph[u]):        # đảo để hàng xóm đầu tiên được lấy ra trước
            if v not in visited:
                st.append(v)
    return thu_tu

print("DFS bằng ngăn xếp:", dfs_ngan_xep(do_thi_dfs, "A"))

DFS bằng ngăn xếp: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS trên lưới: tô loang (flood fill) và đếm vùng

Lưới là đồ thị ẩn: mỗi ô nối với 4 ô kề. DFS từ một ô sẽ "loang" khắp vùng liên thông chứa nó. Đếm số lần phải bắt đầu DFS mới = số vùng.

In [13]:
def dem_dao(grid):
    g = [list(row) for row in grid]
    R, C = len(g), len(g[0])
    def loang(r, c):
        if r < 0 or r >= R or c < 0 or c >= C or g[r][c] != "1":
            return                              # ra ngoài / nước / đã thăm
        g[r][c] = "#"                           # đánh dấu đã thăm
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            loang(r + dr, c + dc)
    dem = 0
    for r in range(R):
        for c in range(C):
            if g[r][c] == "1":
                dem += 1
                loang(r, c)
    return dem, ["".join(row) for row in g]

print(dem_dao(["11000",
               "11010",
               "00011"]))

(2, ['##000', '##0#0', '000##'])


### DFS trên cây: mang thông tin dọc đường đi

Trên cây không cần `visited` (không có chu trình). DFS hay **truyền xuống** thông tin của đường đi từ gốc (tổng, đường đi hiện tại) và **trả lên** kết quả của cây con.

In [14]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_dfs = TreeNode(5, TreeNode(4, TreeNode(11, TreeNode(7), TreeNode(2))), TreeNode(8, TreeNode(13), TreeNode(4)))

def moi_duong_goc_la(node, duong, out):
    if node is None:
        return
    duong.append(node.val)                     # đi xuống: thêm nút
    if node.left is None and node.right is None:
        out.append((list(duong), sum(duong)))  # tới lá: ghi lại một bản sao
    moi_duong_goc_la(node.left, duong, out)
    moi_duong_goc_la(node.right, duong, out)
    duong.pop()                                # quay lên: bỏ nút

ket_qua = []
moi_duong_goc_la(cay_dfs, [], ket_qua)
for d, s in ket_qua:
    print(d, "tổng =", s)

[5, 4, 11, 7] tổng = 27
[5, 4, 11, 2] tổng = 22
[5, 8, 13] tổng = 26
[5, 8, 4] tổng = 17


### So sánh BFS và DFS

| | BFS | DFS |
|---|---|---|
| Cấu trúc | hàng đợi | ngăn xếp / đệ quy |
| Thứ tự | theo lớp (gần → xa) | theo nhánh (sâu trước) |
| Đường ngắn nhất (không trọng số) | **có** | không đảm bảo |
| Bộ nhớ | độ rộng lớn nhất | độ sâu lớn nhất |
| Hợp với | khoảng cách, theo tầng | liệt kê đường đi, vùng liên thông, quay lui |

Lỗi hay gặp: quên `visited` trên đồ thị có chu trình (lặp vô hạn); đệ quy quá sâu trên lưới lớn; ghi `out.append(duong)` thay vì bản sao `list(duong)` (về sau list bị sửa).

## 2.5 Áp dụng vào bài này

Sao chép **sâu** một **đồ thị** dạng đối tượng nút (có chu trình). Duyệt bằng **BFS** hoặc **DFS**; **hash map** `cũ → mới` vừa là bảng `visited`, vừa đảm bảo mỗi nút chỉ được tạo bản sao một lần và các cạnh trỏ đúng bản sao.

# 3. Nền tảng

In [15]:
from typing import List, Optional
from collections import deque
import random, sys
sys.setrecursionlimit(5000)


class Node:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []


def build_graph(adj: List[List[int]]) -> Optional[Node]:
    if not adj:
        return None
    nodes = [Node(i + 1) for i in range(len(adj))]
    for i, nbrs in enumerate(adj):
        nodes[i].neighbors = [nodes[j - 1] for j in nbrs]
    return nodes[0]


def to_adj(node: Optional[Node]) -> List[List[int]]:
    if not node:
        return []
    seen, q = {node.val: node}, deque([node])
    while q:
        cur = q.popleft()
        for nb in cur.neighbors:
            if nb.val not in seen:
                seen[nb.val] = nb
                q.append(nb)
    return [[nb.val for nb in seen[v].neighbors] for v in range(1, len(seen) + 1)]


def all_nodes(node):
    if not node:
        return []
    seen, stack = {id(node): node}, [node]
    while stack:
        cur = stack.pop()
        for nb in cur.neighbors:
            if id(nb) not in seen:
                seen[id(nb)] = nb
                stack.append(nb)
    return list(seen.values())

g = build_graph([[2, 4], [1, 3], [2, 4], [1, 3]])
print(to_adj(g))

[[2, 4], [1, 3], [2, 4], [1, 3]]


`to_adj` đi BFS để đọc lại danh sách kề; `all_nodes` trả về mọi đối tượng nút đạt được — dùng để kiểm tra bản sao **không chia sẻ** nút nào với bản gốc.

# 4. Vì sao không thể "sao chép ngây thơ"?

Đệ quy "tạo nút mới, sao chép từng láng giềng" sẽ **lặp vô hạn** ngay khi có chu trình: `1 → 2 → 1 → 2 → …`

In [16]:
def wrong_naive_clone(node):
    if not node:
        return None
    copy = Node(node.val)
    copy.neighbors = [wrong_naive_clone(nb) for nb in node.neighbors]   # SAI: không nhớ nút đã sao chép
    return copy

try:
    wrong_naive_clone(build_graph([[2], [1]]))
except RecursionError:
    print("RecursionError: đi vòng 1 -> 2 -> 1 -> ... mãi mãi")

RecursionError: đi vòng 1 -> 2 -> 1 -> ... mãi mãi


Ngay cả khi không có chu trình, cách này còn tạo **nhiều bản sao** cho cùng một nút (nếu nút đó có nhiều đường đi tới) — bản sao sẽ không còn cùng cấu trúc.

# 5. Ý tưởng chính: hash map `old → new`

Giữ `clones: dict[Node cũ → Node mới]`:

- Lần đầu gặp nút cũ: tạo nút mới, **ghi vào map ngay**, rồi mới xử lý láng giềng.
- Gặp lại (do chu trình hoặc nhiều đường): trả về bản sao đã có.

```text
clone(node):
    if node in clones: return clones[node]
    copy = Node(node.val)
    clones[node] = copy                     # GHI TRƯỚC khi đệ quy
    for nb in node.neighbors:
        copy.neighbors.append(clone(nb))
    return copy
```

Ghi vào map **trước** khi đệ quy là điều cắt đứt vòng lặp vô hạn.

In [17]:
def clone_dfs(node: Optional[Node]) -> Optional[Node]:
    clones = {}
    def clone(cur):
        if cur in clones:
            return clones[cur]
        copy = Node(cur.val)
        clones[cur] = copy
        for nb in cur.neighbors:
            copy.neighbors.append(clone(nb))
        return copy
    return clone(node) if node else None

g = build_graph([[2, 4], [1, 3], [2, 4], [1, 3]])
c = clone_dfs(g)
print(to_adj(c), "| cùng đối tượng gốc?", c is g)

[[2, 4], [1, 3], [2, 4], [1, 3]] | cùng đối tượng gốc? False


# 6. Mô phỏng

In [18]:
def trace_clone(adj):
    node = build_graph(adj)
    clones = {}
    def clone(cur, depth):
        ind = "  " * depth
        if cur in clones:
            print(f"{ind}gặp lại {cur.val} -> dùng bản sao có sẵn")
            return clones[cur]
        copy = Node(cur.val)
        clones[cur] = copy
        print(f"{ind}tạo bản sao {cur.val}, láng giềng {[nb.val for nb in cur.neighbors]}")
        for nb in cur.neighbors:
            copy.neighbors.append(clone(nb, depth + 1))
        return copy
    c = clone(node, 0)
    print("Danh sách kề bản sao:", to_adj(c))
    return c

_ = trace_clone([[2, 4], [1, 3], [2, 4], [1, 3]])

tạo bản sao 1, láng giềng [2, 4]
  tạo bản sao 2, láng giềng [1, 3]
    gặp lại 1 -> dùng bản sao có sẵn
    tạo bản sao 3, láng giềng [2, 4]
      gặp lại 2 -> dùng bản sao có sẵn
      tạo bản sao 4, láng giềng [1, 3]
        gặp lại 1 -> dùng bản sao có sẵn
        gặp lại 3 -> dùng bản sao có sẵn
  gặp lại 4 -> dùng bản sao có sẵn
Danh sách kề bản sao: [[2, 4], [1, 3], [2, 4], [1, 3]]


# 7. Cách 2: BFS

Tạo bản sao khi **đưa vào hàng đợi**; khi lấy một nút ra, nối các láng giềng (tạo bản sao nếu chưa có).

In [19]:
def clone_bfs(node: Optional[Node]) -> Optional[Node]:
    if not node:
        return None
    clones = {node: Node(node.val)}
    q = deque([node])
    while q:
        cur = q.popleft()
        for nb in cur.neighbors:
            if nb not in clones:
                clones[nb] = Node(nb.val)
                q.append(nb)
            clones[cur].neighbors.append(clones[nb])
    return clones[node]

print(to_adj(clone_bfs(build_graph([[2, 4], [1, 3], [2, 4], [1, 3]]))))

[[2, 4], [1, 3], [2, 4], [1, 3]]


BFS không đệ quy → không lo giới hạn ngăn xếp (với 100 nút thì DFS cũng ổn, nhưng BFS an toàn cho đồ thị lớn).

# 8. Phiên bản đúng format LeetCode

In [20]:
class Solution:
    def cloneGraph(self, node: Optional['Node']) -> Optional['Node']:
        if not node:
            return None
        clones = {node: Node(node.val)}
        q = deque([node])
        while q:
            cur = q.popleft()
            for nb in cur.neighbors:
                if nb not in clones:
                    clones[nb] = Node(nb.val)
                    q.append(nb)
                clones[cur].neighbors.append(clones[nb])
        return clones[node]

In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS  {actual}")


def is_deep_copy(orig, cp):
    """Cùng cấu trúc, và không có nút nào dùng chung."""
    if to_adj(orig) != to_adj(cp):
        return False
    return not ({id(x) for x in all_nodes(orig)} & {id(x) for x in all_nodes(cp)})


def random_connected_adj(n):
    if n == 0:
        return []
    edges = set()
    for v in range(2, n + 1):                       # cây khung ngẫu nhiên để liên thông
        u = random.randint(1, v - 1)
        edges.add((min(u, v), max(u, v)))
    for _ in range(random.randint(0, n * 2)):
        u, v = random.sample(range(1, n + 1), 2) if n > 1 else (1, 1)
        if u != v:
            edges.add((min(u, v), max(u, v)))
    adj = [[] for _ in range(n)]
    for u, v in sorted(edges):
        adj[u - 1].append(v); adj[v - 1].append(u)
    return [sorted(a) for a in adj]


sol = Solution()
check("Ví dụ 1", to_adj(sol.cloneGraph(build_graph([[2, 4], [1, 3], [2, 4], [1, 3]]))), [[2, 4], [1, 3], [2, 4], [1, 3]])
check("Ví dụ 2", to_adj(sol.cloneGraph(build_graph([[]]))), [[]])
check("Ví dụ 3", sol.cloneGraph(None), None)

random.seed(133)
for _ in range(2000):
    adj = random_connected_adj(random.randint(1, 12))
    for f in (sol.cloneGraph, clone_dfs):
        g = build_graph(adj)
        c = f(g)
        assert is_deep_copy(g, c) and to_adj(c) == adj
print("2000 đồ thị liên thông ngẫu nhiên: bản sao đúng cấu trúc, không chia sẻ nút")

Ví dụ 1            PASS  [[2, 4], [1, 3], [2, 4], [1, 3]]


Ví dụ 2            PASS  [[]]
Ví dụ 3            PASS  None


2000 đồ thị liên thông ngẫu nhiên: bản sao đúng cấu trúc, không chia sẻ nút


# 9. Vì sao đúng? Độ phức tạp?

- Mỗi nút cũ được tạo bản sao **đúng một lần** (lần đầu gặp), nhờ map.
- Với mỗi cạnh có hướng `cur → nb` trong danh sách kề, ta thêm đúng một cạnh `clones[cur] → clones[nb]`, **theo đúng thứ tự** trong `cur.neighbors`.
- Đồ thị liên thông → BFS/DFS từ `node` thăm hết mọi nút.

```text
Time:  O(V + E)
Space: O(V) cho map và hàng đợi / ngăn xếp
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Không nhớ nút đã sao chép

Như mục 4 — lặp vô hạn với chu trình.

## Lỗi 2: Ghi vào map **sau** khi đệ quy

In [22]:
def wrong_memo_after(node):
    clones = {}
    def clone(cur):
        if cur in clones:
            return clones[cur]
        copy = Node(cur.val)
        copy.neighbors = [clone(nb) for nb in cur.neighbors]   # đệ quy trước
        clones[cur] = copy                                      # SAI: ghi quá muộn
        return copy
    return clone(node)

try:
    wrong_memo_after(build_graph([[2], [1]]))
except RecursionError:
    print("RecursionError: map chưa có nút 1 khi đệ quy quay lại nó")

RecursionError: map chưa có nút 1 khi đệ quy quay lại nó


## Lỗi 3: Nối cạnh tới nút **cũ**

In [23]:
def wrong_link_old(node):
    clones = {node: Node(node.val)}
    q = deque([node])
    while q:
        cur = q.popleft()
        for nb in cur.neighbors:
            if nb not in clones:
                clones[nb] = Node(nb.val)
                q.append(nb)
            clones[cur].neighbors.append(nb)        # SAI: phải là clones[nb]
    return clones[node]

g = build_graph([[2, 4], [1, 3], [2, 4], [1, 3]])
c = wrong_link_old(g)
print("Danh sách kề giống?", to_adj(c) == to_adj(g), "| nhưng là bản sao sâu?", is_deep_copy(g, c))

Danh sách kề giống? True | nhưng là bản sao sâu? False


Bản sai **trông** đúng khi đọc danh sách kề (vì giá trị giống nhau), nhưng các cạnh trỏ ngược về đồ thị gốc. LeetCode kiểm tra điều này và báo sai.

## Lỗi 4: Dùng `val` làm khóa khi giá trị có thể trùng

Ở bài này `val` phân biệt nên dùng `val` làm khóa cũng được. Nhưng tổng quát (như #138), nên dùng **chính đối tượng nút** làm khóa.

# 11. Mẹo nhận diện pattern

**Sao chép sâu cấu trúc có thể có chu trình / chia sẻ** → map `old → new`, tạo bản sao **trước** khi xử lý các con trỏ.

Bài liên quan: **#138 Copy List with Random Pointer** (cùng ý tưởng trên danh sách), **#1490 Clone N-ary Tree**, **#1485 Clone Binary Tree With Random Pointer**. Trong Python, `copy.deepcopy` cũng dùng một `memo` dict như vậy.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `adjList = [[2, 3], [1, 3], [1, 2]]` (tam giác), mô phỏng DFS clone: nút nào được tạo trước, khi nào "gặp lại"?

## Bài 2 — Điền chỗ trống (DFS)

In [24]:
def exercise_2_clone_graph(node: Optional[Node]) -> Optional[Node]:
    if not node:
        return None
    clones = {}
    def clone(cur):
        # TODO: trả bản sao có sẵn; tạo bản sao, ghi map TRƯỚC, rồi nối láng giềng
        pass
    return clone(node)


# g = build_graph([[2, 4], [1, 3], [2, 4], [1, 3]]); c = exercise_2_clone_graph(g); assert is_deep_copy(g, c)
# assert exercise_2_clone_graph(None) is None

## Bài 3 — Biến thể: sao chép đồ thị **có hướng**, có thể không liên thông

Cho danh sách **mọi** nút (đồ thị có hướng, có thể nhiều thành phần), trả về danh sách bản sao theo cùng thứ tự.

In [25]:
def exercise_3_clone_all(nodes: List[Node]) -> List[Node]:
    # TODO
    pass


# n1, n2, n3 = Node(1), Node(2), Node(3); n1.neighbors = [n2]; n2.neighbors = [n1]; n3.neighbors = [n3]; out = exercise_3_clone_all([n1, n2, n3]); assert [x.val for x in out] == [1, 2, 3] and out[0].neighbors[0] is out[1] and out[2].neighbors[0] is out[2] and out[2] is not n3

## Bài 4 — Biến thể: kiểm tra hai đồ thị có giống nhau (theo nhãn) không

Viết `same_graph(a, b)` trả `True` nếu hai đồ thị (cho bởi một nút) có cùng tập nhãn và cùng danh sách kề cho mỗi nhãn.

In [26]:
def exercise_4_same_graph(a: Optional[Node], b: Optional[Node]) -> bool:
    # TODO
    pass


# assert exercise_4_same_graph(build_graph([[2], [1]]), build_graph([[2], [1]])) is True
# assert exercise_4_same_graph(build_graph([[2, 3], [1], [1]]), build_graph([[2], [1, 3], [2]])) is False
# assert exercise_4_same_graph(None, None) is True

## Bài 5 — Mini challenge: Clone Binary Tree With Random Pointer

Mỗi nút cây có `left`, `right` và `random` (trỏ tới nút bất kỳ trong cây hoặc `None`). Sao chép sâu.

In [27]:
class RNode:
    def __init__(self, val=0, left=None, right=None, random=None):
        self.val, self.left, self.right, self.random = val, left, right, random


def exercise_5_copy_random_tree(root: Optional[RNode]) -> Optional[RNode]:
    # TODO
    pass


# rt = RNode(1, RNode(2), RNode(3)); rt.random = rt.right; rt.left.random = rt; rt.right.random = rt.right; cp = exercise_5_copy_random_tree(rt); assert cp is not rt and cp.random is cp.right and cp.left.random is cp and cp.right.random is cp.right and cp.left is not rt.left

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_clone` sau khi tự làm.

**Bài 3**
- Mức 1: một map dùng chung; gọi `clone(n)` cho từng nút trong danh sách.

**Bài 4**
- Mức 1: so sánh `to_adj(a) == to_adj(b)` (với nhãn 1..n liên tục). Tổng quát hơn: BFS dựng `dict nhãn → list nhãn láng giềng` rồi so.

**Bài 5**
- Mức 1: map `old → new`; DFS qua `left`, `right`, `random` — mỗi con trỏ đều đi qua hàm `clone`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [28]:
_ = trace_clone([[2, 3], [1, 3], [1, 2]])

tạo bản sao 1, láng giềng [2, 3]
  tạo bản sao 2, láng giềng [1, 3]
    gặp lại 1 -> dùng bản sao có sẵn
    tạo bản sao 3, láng giềng [1, 2]
      gặp lại 1 -> dùng bản sao có sẵn
      gặp lại 2 -> dùng bản sao có sẵn
  gặp lại 3 -> dùng bản sao có sẵn
Danh sách kề bản sao: [[2, 3], [1, 3], [1, 2]]


## Đáp án Bài 2

In [29]:
def answer_2_clone_graph(node: Optional[Node]) -> Optional[Node]:
    return clone_dfs(node)

g = build_graph([[2, 4], [1, 3], [2, 4], [1, 3]]); c = answer_2_clone_graph(g); assert is_deep_copy(g, c)
assert answer_2_clone_graph(None) is None
print("Bài 2 PASS (giống clone_dfs ở mục 5)")

Bài 2 PASS (giống clone_dfs ở mục 5)


## Đáp án Bài 3

In [30]:
def answer_3_clone_all(nodes: List[Node]) -> List[Node]:
    clones = {}
    def clone(cur):
        if cur in clones:
            return clones[cur]
        copy = Node(cur.val)
        clones[cur] = copy
        for nb in cur.neighbors:
            copy.neighbors.append(clone(nb))
        return copy
    return [clone(n) for n in nodes]

n1, n2, n3 = Node(1), Node(2), Node(3); n1.neighbors = [n2]; n2.neighbors = [n1]; n3.neighbors = [n3]; out = answer_3_clone_all([n1, n2, n3]); assert [x.val for x in out] == [1, 2, 3] and out[0].neighbors[0] is out[1] and out[2].neighbors[0] is out[2] and out[2] is not n3
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [31]:
def answer_4_same_graph(a: Optional[Node], b: Optional[Node]) -> bool:
    def labels(node):
        if not node:
            return {}
        seen, q, out = {node}, deque([node]), {}
        while q:
            cur = q.popleft()
            out[cur.val] = [nb.val for nb in cur.neighbors]
            for nb in cur.neighbors:
                if nb not in seen:
                    seen.add(nb); q.append(nb)
        return out
    return labels(a) == labels(b)

assert answer_4_same_graph(build_graph([[2], [1]]), build_graph([[2], [1]])) is True
assert answer_4_same_graph(build_graph([[2, 3], [1], [1]]), build_graph([[2], [1, 3], [2]])) is False
assert answer_4_same_graph(None, None) is True
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [32]:
def answer_5_copy_random_tree(root: Optional[RNode]) -> Optional[RNode]:
    clones = {}
    def clone(cur):
        if cur is None:
            return None
        if cur in clones:
            return clones[cur]
        copy = RNode(cur.val)
        clones[cur] = copy
        copy.left = clone(cur.left)
        copy.right = clone(cur.right)
        copy.random = clone(cur.random)
        return copy
    return clone(root)

rt = RNode(1, RNode(2), RNode(3)); rt.random = rt.right; rt.left.random = rt; rt.right.random = rt.right; cp = answer_5_copy_random_tree(rt); assert cp is not rt and cp.random is cp.right and cp.left.random is cp and cp.right.random is cp.right and cp.left is not rt.left
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Map `old → new`. Lần đầu gặp nút: tạo bản sao và **ghi vào map ngay**, rồi nối các láng giềng qua map (tạo bản sao nếu chưa có). Gặp lại thì dùng bản sao cũ. `O(V + E)`.

---

## Checklist trước khi nộp

- [ ] Ghi vào map **trước** khi xử lý láng giềng?
- [ ] Cạnh nối tới **bản sao**, không phải nút cũ?
- [ ] `node = None` → `None`?
- [ ] Giữ đúng thứ tự láng giềng?